In [ ]:
import os
import io
import uuid
import torch
from flask import Flask, request, jsonify
from label_studio_ml.model import LabelStudioMLBase
from google.cloud import texttospeech_v1beta1 as texttospeech
from google.cloud import storage
import os
from google.cloud import texttospeech
from datasets import load_from_disk

In [2]:
#from dotenv import load_dotenv
#load_dotenv() # probably unnecessary for future runs

In [3]:
PROJECT_ID = os.getenv("GOOGLE_CLOUD_PROJECT")

In [4]:
client = texttospeech.TextToSpeechClient()

In [5]:
voices = client.list_voices(language_code="en-US").voices

In [33]:
voices[0]

language_codes: "en-US"
name: "Achernar"
ssml_gender: FEMALE
natural_sample_rate_hertz: 24000

In [ ]:
# IMPORTANT: Set this environment variable to the path of your Google Cloud service account key JSON file
# For local development, you might set this directly or through your shell:
# export GOOGLE_APPLICATION_CREDENTIALS="/path/to/your/service-account-key.json"
# Ensure the service account has "Cloud Text-to-Speech Synthesizer" and "Storage Object Admin" roles.
try:
    if "GOOGLE_APPLICATION_CREDENTIALS" not in os.environ:
        print("WARNING: GOOGLE_APPLICATION_CREDENTIALS environment variable not set.")
        print("Please set it to the path of your Google Cloud service account key JSON file.")
        # As a fallback for local testing, you might hardcode the path, but it's not recommended for production:
        # os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/path/to/your/service-account-key.json"
    
    tts_client = texttospeech.TextToSpeechClient()
    print("Google Cloud TTSclients initialized successfully.")
except Exception as e:
    print(f"Error initializing Google Cloud clients: {e}")
    print("Please ensure your GOOGLE_APPLICATION_CREDENTIALS are correctly set and the service account has necessary permissions.")
    # Exit or handle gracefully depending on your application's needs
    tts_client = None

# google-cloud-texttospeech minimum version 2.29.0 is required.

def synthesize(prompt: str, text: str, output_filepath: str = "output.mp3"):
    """Synthesizes speech from the input text and saves it to an MP3 file.

    Args:
        prompt: Styling instructions on how to synthesize the content in
          the text field.
        text: The text to synthesize.
        output_filepath: The path to save the generated audio file.
          Defaults to "output.mp3".
    """
    client = texttospeech.TextToSpeechClient()

    synthesis_input = texttospeech.SynthesisInput(text=text, prompt=prompt)

    # Select the voice you want to use.
    voice = texttospeech.VoiceSelectionParams(
        language_code="en-US",
        name="Charon",  # Example voice, adjust as needed
        model_name="gemini-2.5-pro-tts"
    )

    audio_config = texttospeech.AudioConfig(
        audio_encoding=texttospeech.AudioEncoding.MP3
    )

    # Perform the text-to-speech request on the text input with the selected
    # voice parameters and audio file type.
    response = client.synthesize_speech(
        input=synthesis_input, voice=voice, audio_config=audio_config
    )

    # The response's audio_content is binary.
    with open(output_filepath, "wb") as out:
        out.write(response.audio_content)
        print(f"Audio content written to file: {output_filepath}")


In [6]:
ds = load_from_disk("/home/pccady/Studies/ThesisProject/data/teanglann/teangl_phon_dataset")

In [7]:
ds[0]

ModuleNotFoundError: No module named 'torch'